# Lab: t-SNE and UMAP

---

In this lab, we will explore two powerful dimensionality reduction techniques: t-Distributed Stochastic Neighbor Embedding (t-SNE) and Uniform Manifold Approximation and Projection (UMAP). These methods are particularly effective for visualizing high-dimensional data in lower dimensions while preserving the local structure of the data. We'll also use PCA as a baseline comparison for evaluating t-SNE and UMAP results.

---

## Data Generation
For this project, we'll use synthetic data that we'll generate using the `make_blobs` function from `sklearn.datasets`. This function allows us to create a dataset with specified characteristics, such as the number of samples, features, centers (clusters), and cluster standard deviation.

But first, let's import the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

import umap.umap_ as UMAP 
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

import plotly.express as px

Now it's time to generate synthetic data with four clusters in a 3D space.

In [ ]:
centers = [[2, -6, -6],
           [-1, 9, 4], 
           [-8, 7, 2], 
           [4, 7, 9]]

# Cluster standard deviations
cluster_std = [1, 1, 2, 3.5]

X, labels_ = make_blobs(n_samples=500, centers=centers, cluster_std=cluster_std, random_state=42)


---

## Data Understanding
Let's start with displaying the data in an interactive Plotly 3D scatter plot.

In [ ]:
df = pd.DataFrame(X, columns=['feature_1', 'feature_2', 'feature_3'])
df['label'] = labels_

fig = px.scatter_3d(df, x='feature_1', y='feature_2', z='feature_3', color='label', opacity=0.7, color_discrete_sequence=px.colors.qualitative.G10, title='3D Scatter Plot of Synthetic Data')
fig.update_traces(marker=dict(size=5, line=dict(width=1, color='DarkSlateGrey')), showlegend=False)
fig.update_layout(coloraxis_showscale=False, width=1000, height=800)  # Remove color bar, resize plot
fig.show()

Let's also create the not interactive version of this plot for Github repository.

In [ ]:
# Let's also create the non-interactive version of this plot for Github repository.
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(df['feature_1'], df['feature_2'], df['feature_3'], c=labels_, cmap='tab10', s=50, alpha=0.7, edgecolor='k')
ax.set_title('3D Scatter Plot of Synthetic Data')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show() 

* 📌 The blobs have varying densities.
* 📌 One blob is distinct from the others.
* 📌 The two largest blobs are distinct from each other, but both have a bit of overlap with the other blob between them.

---

## Data Preparation
Before applying t-SNE and UMAP, we need to standardize the data to ensure that each feature contributes equally to the distance calculations. We'll use `StandardScaler` from `sklearn.preprocessing` for this purpose.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


---

## Modeling and Evaluation

### t-SNE
Let's first apply t-SNE to reduce the dimensionality of our standardized data from 3D to 2D and visualize the results.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter = 1000)
X_tsne = tsne.fit_transform(X_scaled)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_, cmap='viridis', s=50, alpha=0.7, edgecolor='k')
ax.set_title("2D t-SNE Projection of 3D Data")
ax.set_xlabel("t-SNE Component 1")
ax.set_ylabel("t-SNE Component 2")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

* 📌 t-SNE projected the data into four distinct clusters, although the original data had some overlap between a few clusters.
* 📌 We can see that some of the points ended up being in the "wrong" cluster, although to be fair, t-SNE has no knowledge of which clusters the points actually belongs to.
* 📌 Two of the blobs are distinct from each other but "gave up" some of their points to the blob they originally had overlapped with.
* 📌 A "perfect" result would not completely separate the overlaps between blobs.
* 📌 Notice the distance between the blobs is consistent with the degree to which they were originally separated.

### UMAP

In [ ]:
umap_model = UMAP.UMAP(n_components=2, random_state=42, min_dist=0.5, spread=1, n_jobs=1)
X_umap = umap_model.fit_transform(X_scaled)

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
ax.scatter(X_umap[:, 0], X_umap[:, 1], c=labels_, cmap='viridis', s=50, alpha=0.7, edgecolor='k')

ax.set_title("2D UMAP Projection of 3D Data")
ax.set_xlabel("UMAP Component 1", )
ax.set_ylabel("UMAP Component 2", )
ax.set_xticks([])
ax.set_yticks([])
plt.show()

* 📌 UMAP correctly projected the data into four partially distinct clusters, with one cluster completely distinct from the others.
* 📌 Unlike t-SNE, it has preserved the connectedness that the original data had with the partially overlapping clusters.
* 📌 Like t-SNE, somoe of the points ended up in the "wrong" cluster. Again, like t-SNE, all the clusters have similar densities.
* 📌 A "perfect" result would not completely separate the overlaps between blobs, because they actually do overlap in the orifinal feature space.
* 📌 The distance between the clusters is again consistent with the defree to which they were originally separated. 

### PCA
Let's now apply PCA to reduce the dimensionality of our standardized data from 3D to 2D and use it as a baseline for comparison with t-SNE and UMAP results.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
fig = plt.figure(figsize=(8, 6))

ax2 = fig.add_subplot(111)
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_, cmap='viridis', s=50, alpha=0.7, edgecolor='k')
ax2.set_title("2D PCA Projection of 3-D Data")
ax2.set_xlabel("PCA 1")
ax2.set_ylabel("PCA 2")
ax2.set_xticks([])
ax2.set_yticks([])
plt.show()

* 📌 PCA faithfully preserved the relative blob densities. It also preserved the relative separation between blobs.
* 📌 The distance between clusters is very consistent with the degree to which they were originally separated.
* 📌 PCA and t-SNE took very little time to complete compared to UMAP.
* 📌 PCA outperformed both t-SNE and UMAP in this experiment. This points to a common tendency to want to implement more advanced algorithms. The default result is not always an improvement over the simpler established methods.

---

**Next:** [Classification Metrics and Evaluation Techniques](../05_evaluating_and_validating_machine_learning_models/01_classification_metrics_and_evaluation_techniques_theory.md)